# Modeling

## Build a simple Baselinemodel

In [ ]:
# importing modules for data preparation
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# importing modules for training
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, roc_auc_score, confusion_matrix

# importing modules for pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# importing modules for hyperparameter optimization and comparison of models
from sklearn.model_selection import validation_curve, learning_curve


In [ ]:
# instantiate model
model = LogisticRegression(max_iter=500, class_weight='balanced')

In [ ]:
# build pipeline
model_baseline = Pipeline([('model', model)])

In [ ]:
# selection features
num_cols = features_train.select_dtypes(include='number').columns
features_train_baseline = features_train[num_cols]
features_test_baseline = features_test[num_cols]

In [ ]:
# fit pipeline on cleaned (and filtered) training set
model_baseline.fit(features_train_baseline, target_train)

In [ ]:
# predict and evaluate on test set

target_pred = model_baseline.predict(features_test_baseline)

print("Precision:", precision_score(target_test, target_pred))
print("Recall:", recall_score(target_test, target_pred))
print("F1:", f1_score(target_test, target_pred))

In [ ]:
# Notes:
# the baseline model shows low overall performance: Precision is very low (0.18), while Recall is comparatively high (0.60)
# the resulting F1‑score of 0.28 confirms that this model serves only as a starting point 
# before applying feature engineering and further optimization

## Feature Engineering

In [ ]:
#  create engineered features

def engineer_features(df):
    """
    Create engineered features for the car auction fraud model.
    Includes date-based features, ratio features, ZIP reduction.
    Removes raw columns that are no longer needed for modeling.
    
    """

    df = df.copy()

    # date features
    df['PurchaseYear'] = df['PurchDate'].dt.year
    df['PurchaseMonth'] = df['PurchDate'].dt.month
    df['PurchaseWeekday'] = df['PurchDate'].dt.weekday

    # odometer normalization
    df['OdoPerYear'] = df['VehOdo'] / (df['VehicleAge'] + 1)

    # MMR-based features
    df['PriceToMMR'] = df['VehBCost'] / df['MMRCurrentRetailCleanPrice']
    df['MMRDelta'] = df['MMRCurrentRetailCleanPrice'] - df['VehBCost']
    df['MMRRatio'] = df['MMRDelta'] / df['MMRCurrentRetailCleanPrice']

    # Warranty features
    df['WarrantyRatio'] = df['WarrantyCost'] / df['VehBCost']

    # reduction
    df['ZIP3'] = df['VNZIP1'].astype(str).str[:3].astype(int)

    # drop raw columns that are no longer needed
    df = df.drop(columns=['PurchDate', 'VNZIP1'])

    return df

In [ ]:
# new features TRAIN
features_train = engineer_features(features_train)

# new features TEST 
features_test = engineer_features(features_test)

In [ ]:
from sklearn.preprocessing import OneHotEncoder


# select columns
num_cols = features_train.select_dtypes(include='number').columns
cat_cols = features_train.select_dtypes(include='object').columns    
    
# One-Hot Encoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

# fit на TRAIN
encoder.fit(features_train[cat_cols])

# final columns
ohe_columns = encoder.get_feature_names(cat_cols)
final_columns = list(num_cols) + list(ohe_columns)
    
    
def encode(df):
    """
    One-Hot Encoding for train and test.
    
    """
    
    # numeric
    df_num = df[num_cols].reset_index(drop=True)

    # transform train/test
    df_ohe = encoder.transform(df[cat_cols])

    ohe_columns = encoder.get_feature_names(cat_cols)

    # final dataframe
    df_cat = pd.DataFrame(df_ohe, columns=ohe_columns)
    df_encoded = pd.concat([df_num, df_cat], axis=1)
   
    return df_encoded

In [ ]:
features_train = encode(features_train)
features_test = encode(features_test)

## Data Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

# instantiate the scaler
scaler = StandardScaler()

## Dimensionality Reduction

In [ ]:
# Dimensionsreduzierung with PCA

from sklearn.decomposition import PCA

# create pipeline
pipe_pca = Pipeline([('scaler', StandardScaler()),
                    ('pca', PCA(n_components=0.90))])

pipe_pca.fit(features_train)

print("Number of components for pca:", len(final_columns))

# number of components after pca
n_comp = pipe_pca.named_steps['pca'].n_components_
print("Number of components after pca:", n_comp)

# explained variance
print(pipe_pca.named_steps['pca'].explained_variance_ratio_)

In [ ]:
# Notes:
# PCA: Dimensionality reduced from 88 to 58 components

## Features Selection

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# train RF 
model_rf = RandomForestClassifier(random_state=42, class_weight='balanced')
pipe_model_rf = Pipeline([('model', model_rf)])

pipe_model_rf.fit(features_train, target_train)

# feature importance
feature_importance_rf = (pd.DataFrame({'feature': final_columns,
                        'importance': pipe_model_rf.named_steps['model'].feature_importances_})
                        .sort_values(by='importance', ascending=False)
                        .reset_index(drop=True))

feature_importance_rf

In [ ]:
# visualisirung feature_importance

fig, ax = plt.subplots(figsize=(8, 18))

# reverse rows so the most important feature is on top
df_plot = feature_importance_rf[::-1]

ax.barh(df_plot['feature'], df_plot['importance'])

ax.set_xlabel("Importance", fontsize=10)
ax.set_ylabel("Feature", fontsize=6)

ax.margins(y=0)
fig.tight_layout()

In [ ]:
# Notes:
# as expected the top drivers include OdoPerYear, VehBCost, VehOdo, and WarrantyRatio / Cost, ZIP3,VehicleAge,
# followed by several MMR‑based pricing indicators;
# lower‑ranked features include specific some make, size and color categories

## Train model

In [ ]:
from sklearn.exceptions import DataConversionWarning
import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)

#useful imports
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, confusion_matrix, classification_report

In [ ]:
# build unoptimized model

model_baseline_unoptimized = LogisticRegression(max_iter=500, class_weight='balanced')

pipe_model_baseline = Pipeline([('scaler', StandardScaler()),
                                ('model', model_baseline_unoptimized)])
pipe_model_baseline.fit(features_train, target_train)

# predictions
target_pred = pipe_model_baseline.predict(features_test)

# evaluation
print("F1-score:", f1_score(target_test, target_pred))
print("\nClassification report:\n", classification_report(target_test, target_pred))


In [ ]:
# tune hyperparameters

# search space
search_space = {'model__C': [0.001, 0.01, 0.1, 1, 10, 100],
                'model__penalty': ['l1', 'l2'],
                'model__solver': ['saga']}

# GridSearchCV
model_grid = GridSearchCV(estimator=pipe_model_baseline,
                            param_grid=search_space,
                            scoring='f1',
                            cv=5,
                            n_jobs=-1)

model_grid.fit(features_train, target_train)

print("Best score:", model_grid.best_score_)
print("Best params:", model_grid.best_params_)

In [ ]:
#best model grid

best_model_grid = model_grid.best_estimator_
target_pred_grid = best_model_grid.predict(features_test)

# evaluation
print("F1-score:", f1_score(target_test, target_pred_grid))
print("\nClassification report:\n", classification_report(target_test, target_pred_grid))

In [ ]:
# Notes:
# the optimized Logistic Regression achieved an F1‑score of 0.37911 on the test set, 
# light below the baseline result (0.37915)

In [ ]:
# basemodel + pca
model_baseline_unoptimized = LogisticRegression(max_iter=500, class_weight='balanced')

pipe_model_pca = Pipeline([('pca', pipe_pca),
                            ('model', model_baseline_unoptimized)])
pipe_model_pca.fit(features_train, target_train)

# predictions
target_pred = pipe_model_pca.predict(features_test)

# evaluation
print("F1-score:", f1_score(target_test, target_pred))
print("\nClassification report:\n", classification_report(target_test, target_pred))

# shut off some annoying warnings
from sklearn.exceptions import DataConversionWarning, UndefinedMetricWarning
import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)
warnings.filterwarnings(action='ignore', category=UndefinedMetricWarning)
warnings.filterwarnings(action='ignore', category=DeprecationWarning)

In [ ]:
# Notes:
# the Logistic Regression mit pca achieved an F1‑score of 0.3613 on the test set, 
# below the baseline result (0.37915)

In [ ]:
# k-Nearest-Neighbors

from sklearn.exceptions import DataConversionWarning, UndefinedMetricWarning
import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)
warnings.filterwarnings(action='ignore', category=UndefinedMetricWarning)
warnings.filterwarnings(action='ignore', category=DeprecationWarning)

from sklearn.neighbors import KNeighborsClassifier
pipe_knn = Pipeline([('std', StandardScaler()),
                     ('knn', KNeighborsClassifier())])
k = np.unique(np.geomspace(1, 100, 6, dtype='int'))

search_space_knn = {'knn__n_neighbors': k,  
                    'knn__weights': ['uniform', 'distance']}
search_space_knn

#model pipeline
model_knn = GridSearchCV(estimator=pipe_knn, 
                         param_grid=search_space_knn, 
                         scoring='f1',
                         cv=3)

model_knn.fit(features_train, target_train)

# predictions
target_pred = model_knn.predict(features_test)

# evaluation
print("F1-score:", f1_score(target_test, target_pred))
print("\nClassification report:\n", classification_report(target_test, target_pred))

In [ ]:
# Notes:
# the Logistic Regression mit pca achieved an F1‑score of 0.3613 on the test set, 
# below the baseline result (0.37915)

## Model selection

In [ ]:
# select model

models = {"Baseline Logistic Regression": pipe_model_baseline,
          "Optimized Logistic Regression": best_model_grid,
          "Random Forest": pipe_model_rf,
          "Basemodel + pca": pipe_model_pca,
          "k-Nearest-Neighbors": model_knn}



for name, model in models.items():
    print("\n\n=== ", name, " ===")
    
    target_pred = model.predict(features_test)
    
    print("F1-Score:", f1_score(target_test, target_pred))
    
    # confusion matrix
    cm = confusion_matrix(target_test, target_pred)
    print("Confusion matrix:")
    print(cm)
    
    sns.heatmap(cm, annot=True, fmt='d')
    plt.show()
    
    print(classification_report(target_test, target_pred,
                                target_names=["good auto", "bad auto"]))

In [ ]:
# Notes:
# among all tested models, the baseline Logistic Regression achieved the highest F1‑score and the most balanced performance 
# on the minority class; the optimized Logistic Regression and Random Forest did not improve the results, 
# so the Random Forest model ist the best choice for precision
# and the baseline model remains the best choice for recall

## Die finale Datenpipeline

In [ ]:
# finale data pipeline
def data_pipeline(df, model):
    """
    Returns predictions
     - cleaning
     - imputing missing values:
     - features engineering
     - encoding categorical columns
     - predict mit Logistic Regression

    Args: 
        filepath: uncleaned DataFrame
        
    Returns:   
       df with predictions

    """

    #  drop target
    if  'IsBadBuy' in df.columns:
        df = df.drop(columns=['IsBadBuy'])
        
    # preprocessing        
    df = clean_data(df)
    df = clean_categorical_features(df)
    df = impute_missing_values(df)
    df = engineer_features(df)
    df = encode(df)
    
    # predict
    preds = model.predict(df)
    probs = model.predict_proba(df)[:, 1]

    df["is_bad_auto"] = preds
    df["prob_bad_auto"] = probs

    return df

In [ ]:
# important is recall
final_model = pipe_model_baseline
probe = data_pipeline(features_test_orig, final_model)
probe.head()

### Model Interpretation

In [ ]:
from sklearn.inspection import permutation_importance

# original model accuracy
target_test_pred = final_model.predict(features_test)
acc_orig = accuracy_score(target_test, target_test_pred)
print('Original accuracy:', acc_orig)

# preparation
perm_importances = []
features_test_perm = features_test.copy()

# select features
for col in features_test.columns:
    
    # mix the data
    col_perm = features_test_perm[col].sample(frac=1, random_state=0)
    col_perm.index = features_test_perm.index
    features_test_perm[col] = col_perm

    # accuracy the model on mixed data
    target_test_pred_perm = final_model.predict(features_test_perm)
    acc_perm = accuracy_score(target_test, target_test_pred_perm)
    
    # accuracy drop
    perm_importances.append(acc_orig - acc_perm)
    
    # returning the original column
    features_test_perm[col] = features_test[col]

# visualisation
plt.style.use('fivethirtyeight')
fig, ax = plt.subplots(figsize=(8, 20))
ax.set_xlabel('Permutation Feature Importance')

perm_importance = pd.Series(perm_importances, index=features_test.columns).sort_values()
perm_importance.plot(kind='barh', ax=ax)

fig.tight_layout()
plt.show()

In [ ]:
# Notes:
# the permutation importance plot shows that the model2 relies most strongly on Warranty and some make, 
# while features like some Color and VehicleAge etc. have only a negative impact on prediction accuracy;
# many features have little or no effect on the model

In [ ]:
from pdpbox import pdp, info_plots
import matplotlib.pyplot as plt


# select feature
features_to_check = ['VehOdo', 'VehicleAge']

for feat in features_to_check:
    pdp_go = pdp.pdp_isolate(model=final_model,   
                            dataset=features_train,
                            model_features=features_train.columns,
                            feature=feat)

    # PDP & ICE
    fig, axes = pdp.pdp_plot(pdp_go,
                             feature_name = feat,  
                             plot_lines = True, 
                             plot_pts_dist = True,
                             center = True)
    plt.show()

In [ ]:
# Notes:
# Das Modell reagiert plausibel: höherer Verschleiß und höheres Alter führen zu höheren Risiken. 
# PDP und ICE bestätigen diesen Zusammenhang modellagnostisch.

### Abschluss des Projekts

In [ ]:
predictions_aim = pd.read_csv('features_aim.csv')
predictions_aim = data_pipeline(predictions_aim, final_model)
predictions_aim.to_csv('predictions_aim1.csv', index=False)
predictions_aim.head()